<a href="https://colab.research.google.com/github/phy6geniuxGH/my_google_colab_implementations/blob/main/EVoC_UMAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import argparse
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_rand_score, silhouette_score


# ---------------------------------------------------------------------------
# Config - single source of truth
# ---------------------------------------------------------------------------

CONFIG = {
    # Synthetic data (mimics MiniLM-L6-v2: 384-d, L2-normalized)
    "n_samples": 5_000,
    "n_features": 384,
    "n_centers": 20,
    "cluster_std": 1.5,
    "random_state": 42,

    # EVoC (CPU only)
    "evoc_n_neighbors": 15,
    "evoc_base_min_cluster_size": 25,
    "evoc_approx_n_clusters": None,   # None = let persistence pick

    # UMAP (used for both viz and HDBSCAN preproc)
    "umap_n_neighbors": 30,
    "umap_min_dist": 0.1,
    "umap_n_components_viz": 2,
    "umap_n_components_preproc": 10,
    "umap_metric": "cosine",          # CPU only; GPU forces Euclidean

    # HDBSCAN baseline
    "hdbscan_min_cluster_size": 25,
    "hdbscan_min_samples": 5,

    # Runtime
    "use_gpu": False,
    "quick": False,
    "output_dir": "./plots",
}


def apply_quick_overrides(config: dict) -> dict:
    """Shrink the dataset for fast debugging runs.

    Args:
        config: Base config dict.

    Returns:
        Copy of config with reduced n_samples and n_centers.
    """
    config = dict(config)
    config["n_samples"] = 1_000
    config["n_centers"] = 8
    return config


# ---------------------------------------------------------------------------
# Backend resolution
# ---------------------------------------------------------------------------

def resolve_backend(use_gpu: bool) -> dict:
    """Resolve UMAP and HDBSCAN backends (GPU via cuML, or CPU).

    EVoC is always CPU and is not affected by this resolver.

    Args:
        use_gpu: If True, attempt to use RAPIDS cuML; fall back to CPU on import error.

    Returns:
        Dict with keys: UMAP (class), HDBSCAN (class), backend ('gpu'|'cpu'), note (str).
    """
    if use_gpu:
        try:
            from cuml.manifold import UMAP as cuUMAP  # type: ignore
            from cuml.cluster import HDBSCAN as cuHDBSCAN  # type: ignore
            return {
                "UMAP": cuUMAP,
                "HDBSCAN": cuHDBSCAN,
                "backend": "gpu",
                "note": "RAPIDS cuML (GPU)",
            }
        except ImportError as e:
            print(f"[WARN] cuML import failed ({e}); falling back to CPU.")

    import umap
    import hdbscan
    return {
        "UMAP": umap.UMAP,
        "HDBSCAN": hdbscan.HDBSCAN,
        "backend": "cpu",
        "note": "umap-learn + hdbscan (CPU)",
    }


def _to_numpy(arr) -> np.ndarray:
    """Convert cupy/cudf or numpy array to numpy reliably."""
    if hasattr(arr, "get"):           # cupy
        return arr.get()
    if hasattr(arr, "to_numpy"):      # cudf
        return arr.to_numpy()
    return np.asarray(arr)


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def make_synthetic_embeddings(config: dict) -> tuple[np.ndarray, np.ndarray]:
    """Generate L2-normalized blob data mimicking sentence-transformer output.

    Args:
        config: Run config with n_samples, n_features, n_centers, cluster_std,
            random_state.

    Returns:
        X: float32 array of shape (n_samples, n_features), L2-normalized rows.
        y_true: int array of ground-truth cluster labels.
    """
    X, y_true = make_blobs(
        n_samples=config["n_samples"],
        n_features=config["n_features"],
        centers=config["n_centers"],
        cluster_std=config["cluster_std"],
        random_state=config["random_state"],
    )
    X = normalize(X, norm="l2", axis=1).astype(np.float32)
    return X, y_true


# ---------------------------------------------------------------------------
# EVoC (CPU only)
# ---------------------------------------------------------------------------

def cluster_with_evoc(X: np.ndarray, config: dict) -> dict:
    """Fit EVoC and harvest labels, hierarchical layers, and membership strengths.

    Args:
        X: Embedding matrix, shape (n_samples, n_features).
        config: Run config with evoc_* keys.

    Returns:
        Dict with: labels, layers, strengths, tree, model, fit_time.
    """
    from evoc import EVoC

    model = EVoC(
        n_neighbors=config["evoc_n_neighbors"],
        base_min_cluster_size=config["evoc_base_min_cluster_size"],
        approx_n_clusters=config["evoc_approx_n_clusters"],
    )
    t0 = time.perf_counter()
    labels = model.fit_predict(X)
    fit_time = time.perf_counter() - t0
    return {
        "labels": np.asarray(labels),
        "layers": model.cluster_layers_,
        "strengths": model.membership_strengths_,
        "tree": model.cluster_tree_,
        "model": model,
        "fit_time": fit_time,
    }


# ---------------------------------------------------------------------------
# UMAP + HDBSCAN baseline (CPU or GPU)
# ---------------------------------------------------------------------------

def cluster_with_umap_hdbscan(X: np.ndarray, config: dict, backend: dict) -> dict:
    """Baseline pipeline: UMAP dimensionality reduction -> HDBSCAN.

    Args:
        X: Embedding matrix, shape (n_samples, n_features).
        config: Run config with umap_* and hdbscan_* keys.
        backend: Output of resolve_backend().

    Returns:
        Dict with: labels, X_reduced, umap_time, hdbscan_time, fit_time.
    """
    is_gpu = backend["backend"] == "gpu"
    # cuML UMAP cosine support varies by version; Euclidean on L2-normalized
    # data is rank-equivalent to cosine and works on both backends.
    metric = "euclidean" if is_gpu else config["umap_metric"]

    t0 = time.perf_counter()
    reducer = backend["UMAP"](
        n_neighbors=config["umap_n_neighbors"],
        min_dist=config["umap_min_dist"],
        n_components=config["umap_n_components_preproc"],
        metric=metric,
        random_state=config["random_state"],
    )
    X_reduced = reducer.fit_transform(X)
    umap_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    clusterer = backend["HDBSCAN"](
        min_cluster_size=config["hdbscan_min_cluster_size"],
        min_samples=config["hdbscan_min_samples"],
    )
    labels = clusterer.fit_predict(X_reduced)
    hdbscan_time = time.perf_counter() - t1

    return {
        "labels": _to_numpy(labels),
        "X_reduced": _to_numpy(X_reduced),
        "umap_time": umap_time,
        "hdbscan_time": hdbscan_time,
        "fit_time": umap_time + hdbscan_time,
    }


def project_with_umap_2d(X: np.ndarray, config: dict, backend: dict) -> np.ndarray:
    """Project embeddings to 2D for plotting (independent of baseline preproc).

    Args:
        X: Embedding matrix, shape (n_samples, n_features).
        config: Run config with umap_* keys.
        backend: Output of resolve_backend().

    Returns:
        2D projection of shape (n_samples, 2), numpy.
    """
    is_gpu = backend["backend"] == "gpu"
    metric = "euclidean" if is_gpu else config["umap_metric"]
    reducer = backend["UMAP"](
        n_neighbors=config["umap_n_neighbors"],
        min_dist=config["umap_min_dist"],
        n_components=config["umap_n_components_viz"],
        metric=metric,
        random_state=config["random_state"],
    )
    return _to_numpy(reducer.fit_transform(X))


# ---------------------------------------------------------------------------
# Evaluation + plotting
# ---------------------------------------------------------------------------

def evaluate(name: str, X: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Print and return clustering metrics: cluster count, noise %, ARI, silhouette.

    Args:
        name: Display name for the algorithm.
        X: Original (high-dim) embeddings used for silhouette computation.
        y_true: Ground-truth labels.
        y_pred: Predicted cluster labels (-1 = noise).

    Returns:
        Dict with: clusters, noise_pct, ari, silhouette.
    """
    mask = y_pred >= 0
    n_clusters = len(np.unique(y_pred[mask])) if mask.any() else 0
    noise_pct = (~mask).mean() * 100
    ari = adjusted_rand_score(y_true, y_pred)
    if mask.sum() > 1 and n_clusters > 1:
        sil = silhouette_score(X[mask], y_pred[mask], metric="cosine")
    else:
        sil = float("nan")
    print(f"  {name:18s} | clusters={n_clusters:3d} | noise={noise_pct:5.1f}% | "
          f"ARI={ari:.3f} | silhouette={sil:.3f}")
    return {"clusters": n_clusters, "noise_pct": noise_pct, "ari": ari, "silhouette": sil}


def plot_three_way(
    X_2d: np.ndarray,
    y_true: np.ndarray,
    y_evoc: np.ndarray,
    y_baseline: np.ndarray,
    save_path: Path,
) -> Path:
    """Three side-by-side scatter plots sharing the same UMAP layout."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    panels = [
        (y_true,     "Ground Truth"),
        (y_evoc,     "EVoC"),
        (y_baseline, "UMAP + HDBSCAN"),
    ]
    for ax, (labels, title) in zip(axes, panels):
        ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="tab20", s=4, alpha=0.7)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return save_path


def plot_evoc_layers(
    X_2d: np.ndarray,
    layers: list,
    save_path: Path,
    max_layers: int = 4,
) -> Path:
    """Plot the first `max_layers` EVoC hierarchical layers on the UMAP layout."""
    n_layers = min(max_layers, len(layers))
    fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 5))
    if n_layers == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        layer = layers[i]
        n_c = len(np.unique(layer[layer >= 0]))
        ax.scatter(X_2d[:, 0], X_2d[:, 1], c=layer, cmap="tab20", s=4, alpha=0.7)
        ax.set_title(f"Layer {i}: {n_c} clusters")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return save_path


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def main(config: dict) -> None:
    """End-to-end pipeline: data -> EVoC + baseline -> metrics -> plots."""
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    backend = resolve_backend(config["use_gpu"])
    print(f"Backend: {backend['note']}")

    # 1. Data
    X, y_true = make_synthetic_embeddings(config)
    print(f"Data:    shape={X.shape}, dtype={X.dtype}, true_clusters={config['n_centers']}")

    # 2. EVoC (CPU)
    print("\n[1/3] EVoC (CPU; first run includes Numba JIT warmup)...")
    evoc_result = cluster_with_evoc(X, config)
    print(f"  fit_time = {evoc_result['fit_time']:.2f}s")

    # 3. Baseline
    print(f"\n[2/3] UMAP + HDBSCAN ({backend['backend'].upper()})...")
    baseline_result = cluster_with_umap_hdbscan(X, config, backend)
    print(f"  fit_time = {baseline_result['fit_time']:.2f}s "
          f"(UMAP={baseline_result['umap_time']:.2f}s, "
          f"HDBSCAN={baseline_result['hdbscan_time']:.2f}s)")

    # 4. Metrics
    print("\n--- Clustering Quality ---")
    evaluate("EVoC",         X, y_true, evoc_result["labels"])
    evaluate("UMAP+HDBSCAN", X, y_true, baseline_result["labels"])

    # 5. 2D layout + plots
    print(f"\n[3/3] UMAP 2D layout for visualization ({backend['backend'].upper()})...")
    X_2d = project_with_umap_2d(X, config, backend)
    p1 = plot_three_way(
        X_2d, y_true, evoc_result["labels"], baseline_result["labels"],
        output_dir / "comparison.png",
    )
    p2 = plot_evoc_layers(X_2d, evoc_result["layers"], output_dir / "evoc_layers.png")
    print(f"  saved: {p1}")
    print(f"  saved: {p2}")

    # 6. Hierarchy summary
    print("\nEVoC Hierarchy (fine -> coarse):")
    for i, layer in enumerate(evoc_result["layers"]):
        n_c = len(np.unique(layer[layer >= 0]))
        n_n = (layer == -1).sum()
        print(f"  Layer {i}: {n_c:3d} clusters, {n_n:4d} noise")


def parse_args() -> argparse.Namespace:
    """CLI: --gpu, --quick, --output."""
    p = argparse.ArgumentParser(description="EVoC vs UMAP+HDBSCAN demo")
    p.add_argument("--gpu", action="store_true",
                   help="Use RAPIDS cuML for UMAP+HDBSCAN (RTX 4060 / CUDA 12)")
    p.add_argument("--quick", action="store_true",
                   help="Smaller dataset for fast debugging")
    p.add_argument("--output", default="./plots",
                   help="Plot output directory (default: ./plots)")
    # In Jupyter/Colab, sys.argv often contains kernel-specific arguments (e.g., -f).
    # Passing an empty list to parse_args prevents it from trying to parse these,
    # allowing the script to run without an 'unrecognized arguments' error.
    # Note: This means command-line arguments like --gpu won't be picked up
    # directly from the cell execution line in Colab in this setup.
    return p.parse_args([])


if __name__ == "__main__":
    args = parse_args()
    cfg = dict(CONFIG)
    cfg["use_gpu"] = args.gpu
    cfg["output_dir"] = args.output
    if args.quick:
        cfg = apply_quick_overrides(cfg)
        cfg["use_gpu"] = args.gpu        # preserve user's GPU choice
        cfg["output_dir"] = args.output  # preserve output choice
    main(cfg)

Backend: umap-learn + hdbscan (CPU)
Data:    shape=(5000, 384), dtype=float32, true_clusters=20

[1/3] EVoC (CPU; first run includes Numba JIT warmup)...
  fit_time = 29.20s

[2/3] UMAP + HDBSCAN (CPU)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  fit_time = 41.85s (UMAP=41.73s, HDBSCAN=0.12s)

--- Clustering Quality ---
  EVoC               | clusters= 26 | noise= 19.2% | ARI=0.671 | silhouette=0.805
  UMAP+HDBSCAN       | clusters= 20 | noise=  0.0% | ARI=1.000 | silhouette=0.930

[3/3] UMAP 2D layout for visualization (CPU)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  saved: plots/comparison.png
  saved: plots/evoc_layers.png

EVoC Hierarchy (fine -> coarse):
  Layer 0:  30 clusters, 1656 noise
  Layer 1:  26 clusters,  959 noise


In [3]:
!pip install evoc umap-learn hdbscan scikit-learn matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 5.6 MB/s eta 0:00:00
